<a href="https://colab.research.google.com/github/Sayak-coder/SIH_26051/blob/main/thermal_energy_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder , MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from xgboost import XGBRFClassifier

In [3]:
df=pd.read_csv('/content/ladakh_thermal_energy_dataset.csv')
df.head()

,latitude,longitude,shelter_volume_m3,wall_material,wall_thickness_cm,glazing_ratio,insulation_r_value,ghi_w_m2,ambient_temp_c,thermal_mass_kj_k,thermal_energy_kwh
0,33.8745,77.1851,115.4,Stone,27.9,0.26,3.07,381.1,-3.3,57426.76,88.03
1,34.4507,77.5419,111.7,Mud_Brick,51.1,0.23,1.20,272.1,8.3,82398.44,52.26
2,34.2320,77.8729,276.6,Mud_Brick,59.9,0.15,1.93,627.0,-20.0,238384.64,132.51
3,34.0987,77.7322,112.4,Rammed_Earth,16.4,0.05,2.97,209.0,-19.8,28823.57,1.50
4,33.6560,77.8066,118.0,Stone,55.4,0.22,2.24,276.3,-2.7,117916.82,59.63


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   latitude            1000 non-null   float64
 1   longitude           1000 non-null   float64
 2   shelter_volume_m3   1000 non-null   float64
 3   wall_material       1000 non-null   object 
 4   wall_thickness_cm   1000 non-null   float64
 5   glazing_ratio       1000 non-null   float64
 6   insulation_r_value  1000 non-null   float64
 7   ghi_w_m2            1000 non-null   float64
 8   ambient_temp_c      1000 non-null   float64
 9   thermal_mass_kj_k   1000 non-null   float64
 10  thermal_energy_kwh  1000 non-null   float64
dtypes: float64(10), object(1)
memory usage: 86.1+ KB


In [6]:
le=LabelEncoder()
df['wall_material']=le.fit_transform(df['wall_material'])

In [8]:
df['wall_material'].unique()

array([3, 1, 2, 0])

In [9]:
df.isnull().sum()

,0
latitude,0
longitude,0
shelter_volume_m3,0
wall_material,0
wall_thickness_cm,0
glazing_ratio,0
insulation_r_value,0
ghi_w_m2,0
ambient_temp_c,0
thermal_mass_kj_k,0


In [11]:
X=df.drop('thermal_energy_kwh',axis=1)
y=df['thermal_energy_kwh']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=45)

In [13]:
minmax_scale=MinMaxScaler()
X_train['ambient_temp_c']=minmax_scale.fit_transform(X_train[['ambient_temp_c']])
X_test['ambient_temp_c']=minmax_scale.fit_transform(X_test[['ambient_temp_c']])

In [15]:
from sklearn.metrics import r2_score
lr_model=LinearRegression()
lr_model.fit(X_train,y_train)
y_pred=lr_model.predict(X_test)
print(r2_score(y_test,y_pred))

0.8638909231541837


In [19]:
from xgboost import XGBRegressor

xgb_model=XGBRegressor()
xgb_model.fit(X_train,y_train)
y_pred=xgb_model.predict(X_test)
print(r2_score(y_test,y_pred))

0.9720417097151198


testing the model with user difened value


In [29]:
!pip install pvlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 53.6 MB/s eta 0:00:00


In [56]:
from pvlib.location import lookup_altitude, Location
from datetime import datetime
from zoneinfo import ZoneInfo

# Get user inputs for building characteristics
latitude_longitude_string = input('latitude,longitude: ')
shelter_volume_m3 = float(input('Shelter Volume: '))
wall_material = input('Wall Material: ')
wall_thickness_cm = float(input('Wall Thickness: '))
glazing_ratio = float(input('Glazing Ratio in %: '))
insulation_r_value = float(input('Insulation Value: '))

# Parse latitude and longitude
latitude_str, longitude_str = latitude_longitude_string.split(',')
latitude = float(latitude_str)
longitude = float(longitude_str)

# Automatically determine altitude
altitude = lookup_altitude(latitude=latitude, longitude=longitude)

# Get current time in IST for GHI calculation
# current_time = datetime.now(ZoneInfo("Asia/Kolkata"))

# times = pd.DatetimeIndex([current_time])
today = date.today()
times=pd.date_range(start=f'{today} 00:00', end=f'{today} 23:59', freq='1h', tz='Asia/Kolkata')

# Create a pvlib Location object
location = Location(latitude, longitude, tz='Asia/Kolkata', altitude=altitude)

# Calculate Global Horizontal Irradiance (GHI) using a clear-sky model
# This gives an estimate for the current time and location under clear-sky conditions.
clearsky_data = location.get_clearsky(times)
ghi_w_m2 = np.max(clearsky_data['ghi'].to_numpy())

# Automatically determine ambient_temp_c and thermal_mass_kj_k from the dataset's mean
# (as a proxy for 'machine calling the value itself' without external APIs or complex models)
if(wall_material == 'Stone'):
  Material_factor=1.5
elif(wall_material == 'Rammed_Earth'):
  Material_factor=1.3
elif(wall_material == 'Mud_Brick'):
  Material_factor=1.2
elif(wall_material == 'Concrete'):
  Material_factor=1.0

if wall_material == 'Concrete':
    noise = np.random.normal(0, 50)
elif wall_material == 'Mud_Brick':
    noise = np.random.normal(0, 500)
elif wall_material == 'Rammed_Earth':
    noise = np.random.normal(0, 300)
elif wall_material == 'Stone':
    noise = np.random.normal(0, 400)

# ambient_temp_c = df['ambient_temp_c'].mean()
ambient_temp_c = 8.0
thermal_mass_kj_k = (shelter_volume_m3 * wall_thickness_cm * Material_factor * 12)+ noise

input_data = pd.DataFrame({
    'latitude': [latitude],
    'longitude': [longitude],
    'shelter_volume_m3': [shelter_volume_m3],
    'wall_material': [wall_material],
    'wall_thickness_cm': [wall_thickness_cm],
    'glazing_ratio': [glazing_ratio],
    'insulation_r_value': [insulation_r_value],
    'ghi_w_m2': [ghi_w_m2],
    'ambient_temp_c': [ambient_temp_c],
    'thermal_mass_kj_k': [thermal_mass_kj_k]
})

# Apply LabelEncoder to 'wall_material'
input_data['wall_material'] = le.transform(input_data['wall_material'])

# Apply MinMaxScaler to 'ambient_temp_c'
input_data['ambient_temp_c'] = minmax_scale.transform(input_data[['ambient_temp_c']])

# Predict using the xgb_model
predicted_energy = xgb_model.predict(input_data)
print(f"Predicted Thermal Energy (kWh): {predicted_energy[0]:.2f}")

latitude,longitude: 34.62753660427372, 78.72376503514941
Shelter Volume: 150
Wall Material: Rammed_Earth
Wall Thickness: 45
Glazing Ratio in %: 0.20
Insulation Value: 3.5
Predicted Thermal Energy (kWh): 230.63
